# Notebook 21 — Analisis Korelasi Pearson: OHLCV + Sentimen vs OHLCV Saja

**Tujuan:** melihat korelasi Pearson antar fitur untuk dua skema fitur yang dipakai
training (`pipeline/feature_engineering/dataset_builder.py`, single source of truth
yang sama dengan produksi — bukan reimplementasi/data sintetis):

- **Model B** — `FEATURE_COLS` = OHLCV + `sentiment_mean` (6 fitur), dimuat via `load_joined()`.
- **Model A** — `OHLCV_FEATURE_COLS` = OHLCV saja (5 fitur), dimuat via `load_ohlcv_only()`.

**Metode:** Pearson correlation (`DataFrame.corr(method="pearson")`) — mengukur hubungan
**linear** antar dua variabel, rentang -1 (linear negatif sempurna) s.d. +1 (linear
positif sempurna), 0 = tidak ada hubungan linear. Cocok untuk cek awal
multikolinearitas antar fitur & hubungan sentimen-harga, TAPI tidak menangkap
hubungan non-linear.

**Catatan cakupan (supaya tidak disalahpahami):** ini korelasi **antar-fitur** (EDA
deskriptif), BUKAN korelasi fitur-vs-label (arah harga) — analisis itu sudah ada
terpisah di `notebooks/11_diagnostic_data_analysis_horizon.ipynb` (point-biserial,
per split train/val/test) dan endpoint API `/data-information/feature-label-correlation`.

## Bagian 1 — Setup & Muat Data

Reuse langsung `load_joined()` / `load_ohlcv_only()` / `FEATURE_COLS` /
`OHLCV_FEATURE_COLS` dari `dataset_builder.py` — data & kolom fitur PERSIS sama
dengan yang dipakai training (`notebook07_common.py`), bukan versi terpisah.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
if (PROJECT_ROOT / "pipeline").exists() is False:
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pipeline.feature_engineering.dataset_builder import (
    FEATURE_COLS,
    OHLCV_FEATURE_COLS,
    load_joined,
    load_ohlcv_only,
)

FIG_DIR = PROJECT_ROOT / "notebooks" / "outputs" / "notebook21"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
sns.set_theme(style="white")

In [ ]:
df_b = load_joined()       # OHLCV + sentiment_mean (Model B)
df_a = load_ohlcv_only()   # OHLCV saja (Model A)

print(f"Model B (OHLCV + sentiment_mean): {len(df_b)} baris | fitur: {FEATURE_COLS}")
print(f"Model A (OHLCV saja)            : {len(df_a)} baris | fitur: {OHLCV_FEATURE_COLS}")
df_b.head()

## Bagian 2 — Korelasi Pearson: OHLCV + `sentiment_mean` (Model B)

In [ ]:
corr_b = df_b[FEATURE_COLS].corr(method="pearson")

print("=== Matriks Korelasi Pearson -- OHLCV + sentiment_mean (Model B) ===")
print(corr_b.round(4))

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr_b, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Korelasi Pearson -- OHLCV + sentiment_mean (Model B)")
plt.tight_layout()
plt.savefig(FIG_DIR / "notebook21_correlation_model_b.png", dpi=110)
plt.show()

## Bagian 3 — Korelasi Pearson: OHLCV Saja (Model A)

In [ ]:
corr_a = df_a[OHLCV_FEATURE_COLS].corr(method="pearson")

print("=== Matriks Korelasi Pearson -- OHLCV saja (Model A) ===")
print(corr_a.round(4))

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_a, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Korelasi Pearson -- OHLCV saja (Model A)")
plt.tight_layout()
plt.savefig(FIG_DIR / "notebook21_correlation_model_a.png", dpi=110)
plt.show()

## Bagian 4 — Ranking Pasangan Fitur & Cek Multikolinearitas

Pasangan fitur diurutkan berdasarkan `|r|` (nilai absolut korelasi) supaya
hubungan terkuat -- positif maupun negatif -- terlihat duluan, lalu ditandai
kandidat multikolinearitas (`|r| > 0.8`, ambang umum di literatur).

In [ ]:
def top_pairs(corr_df: pd.DataFrame) -> pd.DataFrame:
    cols = corr_df.columns
    rows = [
        {"fitur_1": cols[i], "fitur_2": cols[j], "pearson_r": corr_df.iloc[i, j]}
        for i in range(len(cols))
        for j in range(i + 1, len(cols))
    ]
    pairs_df = pd.DataFrame(rows)
    return pairs_df.reindex(pairs_df["pearson_r"].abs().sort_values(ascending=False).index).reset_index(drop=True)


pairs_b = top_pairs(corr_b)
pairs_a = top_pairs(corr_a)

print("=== Pasangan fitur Model B (OHLCV+sentimen), diurutkan |r| tertinggi ===")
print(pairs_b.to_string(index=False))

print("\n=== Pasangan fitur Model A (OHLCV saja), diurutkan |r| tertinggi ===")
print(pairs_a.to_string(index=False))

MULTICOLLINEARITY_THRESHOLD = 0.8
print(f"\n=== Kandidat multikolinearitas (|r| > {MULTICOLLINEARITY_THRESHOLD}) ===")
for name, pairs_df in [("Model B", pairs_b), ("Model A", pairs_a)]:
    flagged = pairs_df[pairs_df["pearson_r"].abs() > MULTICOLLINEARITY_THRESHOLD]
    if flagged.empty:
        print(f"  {name}: tidak ada pasangan di atas ambang")
    else:
        print(f"  {name}:")
        print(flagged.to_string(index=False))

pairs_b.to_csv(FIG_DIR / "notebook21_correlation_pairs_model_b.csv", index=False)
pairs_a.to_csv(FIG_DIR / "notebook21_correlation_pairs_model_a.csv", index=False)

In [ ]:
print("=== Korelasi sentiment_mean vs tiap fitur harga/volume (khusus Model B) ===")
sentiment_corr = corr_b["sentiment_mean"].drop("sentiment_mean")
print(sentiment_corr.reindex(sentiment_corr.abs().sort_values(ascending=False).index))

## Ringkasan

- Kolom `Open/High/Low/Close` secara definisi berasal dari harga hari yang sama,
  jadi korelasi Pearson yang sangat tinggi antar sesama fitur harga (mendekati
  ±1) **diekspektasikan** — bukan temuan baru, dan **bukan** alasan untuk drop
  fitur (LSTM tetap butuh keempatnya sebagai representasi rentang harga harian).
- `Volume` diekspektasikan berkorelasi lemah dengan fitur harga (mekanisme
  pasar berbeda: level harga vs jumlah transaksi).
- `sentiment_mean` diekspektasikan berkorelasi lemah dengan fitur harga/volume —
  konsisten dengan cakupan berita yang tidak selalu match hari bursa & sifat
  sentimen sebagai sinyal independen (bukan turunan harga).
- **Keterbatasan:** Pearson hanya menangkap hubungan **linear** pada titik waktu
  yang sama (lag 0) — TIDAK mengukur hubungan lag/non-linear. Untuk analisis
  lead-lag sentimen-harga, lihat `notebooks/11_diagnostic_data_analysis_horizon.ipynb`
  Bagian F (cross-correlation lag 0..10 hari).

Artefak tersimpan di `notebooks/outputs/notebook21/`:
`notebook21_correlation_model_{a,b}.png`, `notebook21_correlation_pairs_model_{a,b}.csv`.